# 06 — End-to-End ANPR Inference

Run YOLO detector + TrOCR recognizer in sequence.
Saves plate crops, prediction CSV, and YOLO confidence scores.

In [ ]:
from pathlib import Path

import cv2
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from ultralytics import YOLO

In [ ]:
IMAGE_DIR   = Path('C:/path/to/private/data/DATASET/val/images')
YOLO_MODEL  = Path('C:/path/to/private/models/stage5_v4_only_best.pt')
TROCR_MODEL = Path('C:/path/to/private/models/trocr_plate_model_v2/best_model')
OUTPUT_DIR  = Path('pipeline_results')
CROP_DIR    = OUTPUT_DIR / 'crops'
OUTPUT_CSV  = OUTPUT_DIR / 'pipeline_predictions.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

yolo = YOLO(str(YOLO_MODEL))
processor = TrOCRProcessor.from_pretrained(str(TROCR_MODEL))
trocr = VisionEncoderDecoderModel.from_pretrained(str(TROCR_MODEL)).to(DEVICE)
trocr.eval()

In [ ]:
def read_plate(crop_bgr):
    """Run TrOCR on a BGR crop; return (text, mean_token_confidence)."""
    pil_image = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    pixel_values = processor(images=pil_image, return_tensors='pt').pixel_values.to(DEVICE)
    with torch.no_grad():
        outputs = trocr.generate(
            pixel_values, max_length=16,
            output_scores=True, return_dict_in_generate=True
        )
    text = processor.batch_decode(outputs.sequences, skip_special_tokens=True)[0].upper().replace(' ', '')
    probs = [torch.softmax(s, dim=-1).max().item() for s in outputs.scores]
    return text, (sum(probs) / len(probs) if probs else 0.0)


image_files = [p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}]
rows = []

for image_path in tqdm(image_files, desc='ANPR inference'):
    frame = cv2.imread(str(image_path))
    if frame is None:
        continue
    plate_text, ocr_conf, yolo_conf = '', 0.0, 0.0
    try:
        detections = yolo.predict(frame, verbose=False)
        boxes = detections[0].boxes
        if len(boxes):
            best = max(boxes, key=lambda b: float(b.conf))
            yolo_conf = float(best.conf)
            x1, y1, x2, y2 = best.xyxy[0].cpu().numpy().astype(int)
            crop = frame[y1:y2, x1:x2]
            if crop.size > 0:
                cv2.imwrite(str(CROP_DIR / (image_path.stem + '_crop.jpg')), crop)
                plate_text, ocr_conf = read_plate(crop)
    except Exception:
        pass
    rows.append({
        'image_name': image_path.name,
        'predicted_plate': plate_text,
        'ocr_confidence': round(ocr_conf, 4),
        'yolo_confidence': round(yolo_conf, 4),
    })

pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False)
print('saved =', OUTPUT_CSV)
print('total_images =', len(rows))